# 08 · Tuning the node2vec embedding (one hop)

Goal: a solid node2vec embedding where each stay's context is its **directly linked stays** (`window=1`). This notebook finds the settings; notebook 07 then uses them.

**How "best" is decided: without the outcomes.** Hide 10% of the links, train node2vec on the remaining 90%, and ask how well the embedding tells the hidden links apart from random non-links (AUROC, 0.5 = no better than chance). That is exactly what a one-hop embedding is trained to do, it needs no labels, and it treats every network the same way. Choosing settings by how well they predict long stay or blood pressure would use the answers to pick the settings, and tuning on the honest network alone would give it a head start over the cheating one.

Two details:

- About a third of the links join two stays of the **same patient**, an easy target. The score used for choosing is on hidden links between **different patients** (`auc_diff`).
- Seed-to-seed noise is bigger than most differences between settings, so every setting runs with several seeds and is chosen with a **one-standard-error rule**: take the simplest setting whose average is within one standard error of the best, and keep the default unless something is clearly better.

Plan: (1) p and q, (2) one-at-a-time sweeps of the other settings around the best p, q, (3) combine, then confirm against the default with more seeds, next to simple non-learned baselines. The outcome score of the finished embedding is a **check only**.

In [1]:
import numpy as np
import pandas as pd

from ehrgraph.gnn.n2v_sweep import (
    PARAM_COLS, combine_winners, heuristic_baseline, label_of, ofat, pick_one_se,
    run_sweep, split_edges, summarize,
)
from ehrgraph.gnn.node_features import _oof_auc, node2vec_block
from ehrgraph.netstats.task_analysis import task_graphs_from_labels
from ehrgraph.project import setup

p = setup()

TASKS = {"guo_los": "long stay", "new_hypertension": "new high blood pressure"}
NAME = {"safe": "honest", "leaky": "cheating", "null": "random"}

# --- cost knobs. Rough estimate on one laptop core: about an hour with these defaults,
# --- about 25 minutes with QUICK = True (long-stay networks only, fewer seeds).
QUICK = False
STAGE1_SEEDS, SEEDS, CONFIRM_SEEDS = (0, 1), (0, 1, 2), (0, 1, 2, 3, 4)
if QUICK:
    STAGE1_SEEDS, SEEDS, CONFIRM_SEEDS = (0, 1), (0, 1), (0, 1, 2)

BASE = dict(dim=64, num_walks=10, walk_length=30, window=1, negatives=5, epochs=5,
            p=1.0, q=1.0, neg_power=1.0, weighted=False)

data folder: /home/ianstedham/network-analysis-ml/trimmed-dataset/ehrshot-patient-graphs/data/omop
results go to: /home/ianstedham/network-analysis-ml/trimmed-dataset/ehrshot-patient-graphs/results/aggregate


## Networks

Only the networks and the stay order are needed here (no analysis), plus the outcomes for the final check. We tune on **both** networks (honest and cheating) of both questions together, so the chosen setting isn't fitted to one of them.

In [2]:
nets, y, groups = {}, {}, {}
for task in TASKS:
    nets[task] = task_graphs_from_labels(p.diagnoses, p.labels(task), task, n_null=1)
    tg = nets[task]
    y[task] = np.array([tg.labels[n] for n in tg.nodes])
    groups[task] = pd.factorize(pd.Series([n[0] for n in tg.nodes]))[0]
    print(f"{TASKS[task]}: {len(tg.nodes):,} stays, {tg.graphs['safe'].number_of_edges():,} honest links")

TARGETS = {f"{TASKS[t]} / {NAME[g]}": (nets[t].graphs[g], nets[t].nodes)
           for t in TASKS for g in ("safe", "leaky") if not (QUICK and t != "guo_los")}
print(list(TARGETS))

long stay: 6,998 stays, 18,581 honest links
new high blood pressure: 3,511 stays, 9,787 honest links
['long stay / honest', 'long stay / cheating', 'new high blood pressure / honest', 'new high blood pressure / cheating']


## Reference: what do simple, non-learned rules score?

Common neighbours and Jaccard overlap use no embedding at all: two stays score high if they share linked stays. These networks are full of closed triangles, so these rules should be strong. A learned embedding has to be judged against them, not only against other settings of itself.

In [3]:
ref = {name: {k: v for k, v in heuristic_baseline(split_edges(g, nodes, 0.1, 0)).items() if k.endswith("diff")}
       for name, (g, nodes) in TARGETS.items()}
display(pd.DataFrame(ref).T.round(3))     # AUROC on hidden links between different patients

,common_neighbors_auc_diff,jaccard_auc_diff
long stay / honest,0.861,0.861
long stay / cheating,0.925,0.925
new high blood pressure / honest,0.859,0.860
new high blood pressure / cheating,0.914,0.915


## Stage 1: p and q

`q` below 1 pushes the walker outward, above 1 keeps it close (the lecture's "large q: local"). `p` below 1 makes it step back. At `window=1` they only change how often each link is sampled, so I expect little effect, especially from `p`; the grid includes `q` above 1, which the earlier sweep never reached.

In [4]:
configs1 = [{"p": p_, "q": q_} for p_ in (0.25, 0.5, 1.0, 2.0, 4.0) for q_ in (0.25, 0.5, 1.0, 2.0, 4.0)]
df1 = run_sweep(configs1, TARGETS, BASE, seeds=STAGE1_SEEDS)
sm1 = summarize(df1)
display(sm1[["score", "se", *TARGETS, "within_1se"]].round(3))

pick1 = pick_one_se(sm1, prefer=label_of({"p": 1.0, "q": 1.0}))     # keep p = q = 1 unless clearly worse
best1 = {"p": float(sm1.loc[pick1, "p"]), "q": float(sm1.loc[pick1, "q"])}
print("stage 1 choice:", best1, "| settings within one standard error of the best:", int(sm1.within_1se.sum()))

[1/25] p=0.25, q=0.25
[2/25] p=0.25, q=0.5
[3/25] p=0.25, q=1.0
[4/25] p=0.25, q=2.0
[5/25] p=0.25, q=4.0
[6/25] p=0.5, q=0.25
[7/25] p=0.5, q=0.5
[8/25] p=0.5, q=1.0
[9/25] p=0.5, q=2.0
[10/25] p=0.5, q=4.0
[11/25] p=1.0, q=0.25
[12/25] p=1.0, q=0.5
[13/25] p=1.0, q=1.0
[14/25] p=1.0, q=2.0
[15/25] p=1.0, q=4.0
[16/25] p=2.0, q=0.25
[17/25] p=2.0, q=0.5
[18/25] p=2.0, q=1.0
[19/25] p=2.0, q=2.0
[20/25] p=2.0, q=4.0
[21/25] p=4.0, q=0.25
[22/25] p=4.0, q=0.5
[23/25] p=4.0, q=1.0
[24/25] p=4.0, q=2.0
[25/25] p=4.0, q=4.0


,score,se,long stay / honest,long stay / cheating,new high blood pressure / honest,new high blood pressure / cheating,within_1se
config,,,,,,,
"p=4.0, q=0.25",0.939,0.001,0.928,0.957,0.928,0.943,True
"p=0.5, q=2.0",0.939,0.000,0.930,0.955,0.922,0.948,True
"p=0.25, q=0.5",0.938,0.001,0.924,0.955,0.927,0.947,True
"p=0.5, q=0.5",0.938,0.001,0.931,0.956,0.920,0.947,True
"p=4.0, q=0.5",0.938,0.000,0.925,0.959,0.924,0.945,True
"p=0.5, q=0.25",0.938,0.002,0.927,0.953,0.922,0.948,False
"p=2.0, q=2.0",0.938,0.001,0.927,0.957,0.921,0.946,False
"p=2.0, q=1.0",0.938,0.001,0.929,0.957,0.920,0.944,False
"p=1.0, q=0.25",0.937,0.001,0.926,0.958,0.922,0.943,False


stage 1 choice: {'p': 4.0, 'q': 0.25} | settings within one standard error of the best: 5


In [5]:
# is there a real p or q effect? (average over everything else)
print(df1.groupby("q").auc_diff.mean().round(3).to_string()); print(); print(df1.groupby("p").auc_diff.mean().round(3).to_string())

q
0.25    0.937
0.50    0.937
1.00    0.936
2.00    0.937
4.00    0.932

p
0.25    0.935
0.50    0.937
1.00    0.935
2.00    0.936
4.00    0.936


## Stage 2: the other settings, one at a time

Around the stage-1 choice, each setting varies alone. `epochs` goes up to 40 because a one-hop embedding has to pass information link by link and may need many passes. `weighted` uses each link's similarity as its sampling strength (the earlier runs treated all links equally). `num_walks` sets how many link samples the embedding sees.

In [6]:
SPACE_FULL = {"dim": [16, 32, 64, 128], "epochs": [5, 10, 20, 40], "negatives": [2, 5, 10, 20],
              "neg_power": [0.75, 1.0], "weighted": [False, True], "num_walks": [10, 20, 40]}
SPACE = {k: [v for v in vs if v != BASE[k]] for k, vs in SPACE_FULL.items()}     # the base value is the reference row

df2 = run_sweep(ofat(best1, SPACE), TARGETS, BASE, seeds=SEEDS)
sm2 = summarize(df2)
display(sm2[["score", "se", *TARGETS, "within_1se", "cost"]].round(3))

FINAL = combine_winners(sm2, best1, BASE, SPACE)
print("combined setting:", FINAL)

[1/14] p=4.0, q=0.25
[2/14] dim=16, p=4.0, q=0.25
[3/14] dim=32, p=4.0, q=0.25
[4/14] dim=128, p=4.0, q=0.25
[5/14] epochs=10, p=4.0, q=0.25
[6/14] epochs=20, p=4.0, q=0.25
[7/14] epochs=40, p=4.0, q=0.25
[8/14] negatives=2, p=4.0, q=0.25
[9/14] negatives=10, p=4.0, q=0.25
[10/14] negatives=20, p=4.0, q=0.25
[11/14] neg_power=0.75, p=4.0, q=0.25
[12/14] p=4.0, q=0.25, weighted=True
[13/14] num_walks=20, p=4.0, q=0.25
[14/14] num_walks=40, p=4.0, q=0.25


,score,se,long stay / honest,long stay / cheating,new high blood pressure / honest,new high blood pressure / cheating,within_1se,cost
config,,,,,,,,
"dim=128, p=4.0, q=0.25",0.942,0.001,0.930,0.957,0.927,0.953,True,1152000
"negatives=2, p=4.0, q=0.25",0.940,0.001,0.929,0.957,0.928,0.947,False,288000
"neg_power=0.75, p=4.0, q=0.25",0.940,0.001,0.928,0.958,0.929,0.946,False,576000
"p=4.0, q=0.25",0.940,0.001,0.929,0.956,0.928,0.946,False,576000
"p=4.0, q=0.25, weighted=True",0.938,0.001,0.928,0.958,0.922,0.945,False,576000
"epochs=10, p=4.0, q=0.25",0.938,0.001,0.927,0.955,0.925,0.944,False,1152000
"negatives=10, p=4.0, q=0.25",0.937,0.001,0.926,0.953,0.926,0.944,False,1056000
"negatives=20, p=4.0, q=0.25",0.937,0.001,0.928,0.953,0.925,0.941,False,2016000
"num_walks=20, p=4.0, q=0.25",0.937,0.000,0.927,0.956,0.919,0.944,False,1152000


combined setting: {'p': 4.0, 'q': 0.25, 'dim': 128}


## Confirm

The combined setting has never been run as a whole, so it is compared with the default and the stage-1 choice using **more seeds**. If it isn't clearly better than the default, use the default.

In [7]:
confirm = {"default": {}, "stage 1 (p, q)": best1, "combined": FINAL}
dfc = run_sweep(list(confirm.values()), TARGETS, BASE, seeds=CONFIRM_SEEDS)
smc = summarize(dfc)
display(smc[["score", "se", *TARGETS, "within_1se"]].round(3))
print("link-prediction reference (different-patient links):"); display(pd.DataFrame(ref).T.round(3))

[1/3] baseline
[2/3] p=4.0, q=0.25
[3/3] dim=128, p=4.0, q=0.25


,score,se,long stay / honest,long stay / cheating,new high blood pressure / honest,new high blood pressure / cheating,within_1se
config,,,,,,,
"dim=128, p=4.0, q=0.25",0.942,0.001,0.932,0.958,0.926,0.951,True
"p=4.0, q=0.25",0.938,0.001,0.927,0.956,0.925,0.945,False
baseline,0.937,0.001,0.928,0.957,0.918,0.944,False


link-prediction reference (different-patient links):


,common_neighbors_auc_diff,jaccard_auc_diff
long stay / honest,0.861,0.861
long stay / cheating,0.925,0.925
new high blood pressure / honest,0.859,0.860
new high blood pressure / cheating,0.914,0.915


### Check only: is the finished embedding still useful for the outcomes?

Out-of-fold AUROC of the embedding alone, folds grouped by patient, on the **full** networks (not the 90% used for tuning). This is **not** used to choose anything. The random network should stay near 0.5.

In [8]:
rows = []
for task in TASKS:
    for g in ("safe", "leaky", "null"):
        for name, cfg in (("default", {}), ("tuned", FINAL)):
            b = node2vec_block(nets[task].graphs[g], nets[task].nodes, **{**BASE, **cfg, "seed": 0})
            rows.append({"question": TASKS[task], "network": NAME[g], "setting": name,
                         "outcome AUROC": _oof_auc(b.values, y[task], groups[task], 0)})
display(pd.DataFrame(rows).pivot_table(index=["question", "network"], columns="setting", values="outcome AUROC").round(3))

setting                           default  tuned
question                network                 
long stay               cheating    0.654  0.639
                        honest      0.677  0.685
                        random      0.507  0.492
new high blood pressure cheating    0.626  0.629
                        honest      0.608  0.613
                        random      0.488  0.492

In [9]:
# save the sweep numbers (aggregate only, no patient data)
pd.concat({"stage1": df1, "stage2": df2, "confirm": dfc}, names=["stage"]).reset_index(level=0) \
  .to_csv(p.results / "n2v_sweep.csv", index=False)
print("FINAL =", {**BASE, **FINAL})

FINAL = {'dim': 128, 'num_walks': 10, 'walk_length': 30, 'window': 1, 'negatives': 5, 'epochs': 5, 'p': 4.0, 'q': 0.25, 'neg_power': 1.0, 'weighted': False}


## Use it

Copy the printed `FINAL` into the `N2V` line of notebook 07 (keep `window=1`), rebuild, and use the same setting for every network. If the combined setting did not beat the default in the confirmation, keep the default.

## What we learned

*(fill in after running)*

- **Did p or q matter?** Compare the `q` and `p` averages in stage 1.
- **Which settings moved the score by more than a standard error?** Probably `epochs`, maybe `dim`; the table says.
- **Did the embedding beat common neighbours?** If not, the learned embedding adds little for predicting links, which is worth stating.
- **Is the tuned embedding still useful for the outcomes, and is the random network still at 0.5?**